# Crash Course — 그 밖의 omics와 multi-omics 통합

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/other-omics.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/other-omics.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

가상의 두 층(같은 시료의 전사체와 단백질체)으로 통합 흐름의 2단계(시료 정합성)와 가장 단순한 층간 분석(유전자별 상관)을 본다. 데이터는 가상이며 생물학적 의미가 없다.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats

rng = np.random.default_rng(7)
n_gene, n_samp = 200, 20
genes = [f"G{i:03d}" for i in range(n_gene)]
ids = [f"S{i:02d}" for i in range(1, n_samp + 1)]

# 가상 데이터: 유전자마다 "mRNA가 단백질을 얼마나 따라가는가"(rho)가 다르다
rho = rng.uniform(-0.1, 0.9, n_gene)
z = rng.normal(size=(n_gene, n_samp))
rna = pd.DataFrame(z, index=genes, columns=ids)
prot = pd.DataFrame(rho[:, None] * z + np.sqrt(1 - rho[:, None] ** 2) * rng.normal(size=z.shape),
                    index=genes, columns=ids)

# 현실처럼 어긋나게 만든다: 단백질 표는 열 순서가 다르고, S07이 빠졌고, RNA에 없는 S21이 있다
order = rng.permutation(ids).tolist()
order.remove("S07")
prot = prot[order].assign(S21=rng.normal(size=n_gene))
prot = prot.iloc[rng.permutation(n_gene)[:150]]      # 단백질은 150개만 정량되었다

# 1) 시료 ID 정합성 확인: 순서가 아니라 이름으로
print("RNA 시료 수:", rna.shape[1], "/ 단백질 시료 수:", prot.shape[1])
print("ID 중복:", rna.columns.duplicated().any(), prot.columns.duplicated().any())
print("RNA에만 있는 시료:", sorted(set(rna.columns) - set(prot.columns)))
print("단백질에만 있는 시료:", sorted(set(prot.columns) - set(rna.columns)))
print("열 순서가 같은가:", list(rna.columns[:5]), "vs", list(prot.columns[:5]))

# 2) 공통 시료·공통 유전자만, 같은 순서로 맞춘다
common_s = sorted(set(rna.columns) & set(prot.columns))
common_g = sorted(set(rna.index) & set(prot.index))
R, P = rna.loc[common_g, common_s], prot.loc[common_g, common_s]
assert (R.columns == P.columns).all() and (R.index == P.index).all()
print(f"\n맞춘 뒤: 유전자 {len(common_g)}개 × 시료 {len(common_s)}개  (변수 수 / 시료 수 = {len(common_g) / len(common_s):.1f})")

# 3) 유전자별 층간 상관 (Spearman): 한 유전자의 mRNA와 단백질이 시료에 걸쳐 함께 움직이는가
r = pd.Series([stats.spearmanr(R.loc[g], P.loc[g])[0] for g in common_g], index=common_g)
print("유전자별 mRNA–단백질 상관 요약:")
print(r.describe()[["min", "25%", "50%", "75%", "max"]].round(2).to_string())
print(f"상관 > 0.5: {(r > 0.5).sum()}개,  상관 < 0.2: {(r < 0.2).sum()}개")

# 4) 순서만 믿고 붙이면: 이름을 무시하고 위치로 짝지은 경우
k = len(common_s)
r_bad = pd.Series([stats.spearmanr(rna.loc[g].to_numpy()[:k], prot.loc[g].to_numpy()[:k])[0] for g in common_g])
print(f"\n위치로 짝지었을 때 상관 중앙값: {r_bad.median():.2f}   (이름으로 맞췄을 때: {r.median():.2f})")